# 12. 최종모형 방법론 감사

최종모형(`10_최종모형`)을 논문에 확정하기 전에, 항목별로 실제 계산해서 점검한다. 답변 문서는 `../방법론_감사.md`.

**기준 모형 (M0)**: 셀 × 노선, 오전 07~09시, Poisson QMLE, offset = log(2024 승차), 노선 FE, 셀 군집 SE.
X = 보행 20분 이내, 20~60분 (기준 60분 초과), IC 도로거리, 경쟁 목적지 비율, 20~50대 인구, 건축물 수

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

import statsmodels.api as sm
from scipy.stats import norm, spearmanr

data = load_model_data()
X_FULL = ["walk_20", "walk_20_60", "ic_access", "competing_share", "pop_20_50", "buildings"]
trips = load_trips().dropna(subset=["셀 ID"])


def build_pairs(hours, keep_zero=False):
    frame = (trips[trips["시"].between(*hours)]
             .pivot_table(index=["셀 ID", "노선"], columns="연도", values="승차인원", aggfunc="sum", fill_value=0)
             .reset_index().rename(columns={2024: "y2024", 2025: "y2025"}))
    frame["셀 ID"] = frame["셀 ID"].astype(int)
    frame = frame.merge(data[["셀 ID", "walk_min", "nearest_station"] + X_FULL], on="셀 ID")
    return frame.reset_index(drop=True) if keep_zero else frame[frame.y2024 > 0].reset_index(drop=True)


morning, allday = build_pairs(MORNING_HOURS), build_pairs(ALL_DAY_HOURS)


def design(frame, x_cols, route_fe=True, station_fe=False):
    parts = [frame[x_cols]]
    if route_fe:
        parts.append(pd.get_dummies(frame.노선, prefix="노선", drop_first=True, dtype=float))
    if station_fe:
        parts.append(pd.get_dummies(frame.nearest_station, prefix="역", drop_first=True, dtype=float))
    return pd.concat(parts, axis=1)


def fit(frame, x_cols, route_fe=True, station_fe=False, cluster="셀 ID"):
    return poisson_rate(frame.y2025, frame.y2024, design(frame, x_cols, route_fe, station_fe),
                        cluster=None if cluster is None else frame[cluster])


def effect(result, name):
    return pd.Series({"변화율 차이(%)": (np.exp(result.params[name]) - 1) * 100, "p": result.pvalues[name]})


m0 = fit(morning, X_FULL)
print(f"M0: N={len(morning)}쌍, 셀 {morning['셀 ID'].nunique()}개, 노선 {morning.노선.nunique()}개")
coef_table(m0, rate=True).loc[[VARIABLE_LABELS[c] for c in X_FULL]]

M0: N=186쌍, 셀 118개, 노선 24개


,coef,p,변화율 차이(%),유의
보행 20분 이내,-0.5755,0.0028,-43.7566,***
보행 20~60분,-0.1887,0.1632,-17.1945,
IC까지 도로거리(km),-0.1139,0.0078,-10.7675,***
GTX 경쟁 목적지 비율(%),-0.0048,0.2932,-0.4788,
2024년 20~50대 인구,-0.0001,0.6775,-0.0066,
2024년 건축물 수,-0.0014,0.6671,-0.1372,


## 1. Poisson QMLE + offset 구조

offset 모형: `log E[y2025] = log y2024 + Xβ` ⇔ `E[y2025 / y2024 | X] = exp(Xβ)`. 즉 **조건부 변화율(비율)을 X의 곱셈 효과로 모형화**한다.
이 해석은 offset 계수가 1이라는 가정에 기대고 있다 → **log(y2024)를 자유 계수로 넣어 1과 같은지 검정**한다(1보다 작으면 평균회귀가 있다는 뜻).

In [2]:
free = poisson_rate(morning.y2025, morning.y2024 * 0 + 1,
                    pd.concat([design(morning, X_FULL), np.log(morning.y2024).rename("log_y2024")], axis=1), cluster=morning["셀 ID"])
theta, se = free.params["log_y2024"], free.bse["log_y2024"]
print(f"log(y2024)의 자유 계수 θ = {theta:.3f} (SE {se:.3f}), H0: θ=1 → z = {(theta-1)/se:.2f}, p = {2*norm.sf(abs((theta-1)/se)):.4f}")
print("θ를 자유롭게 둔 모형의 핵심 계수:")
display(pd.DataFrame({n: effect(free, n) for n in ["walk_20", "walk_20_60", "ic_access"]}).T.round(4))

log(y2024)의 자유 계수 θ = 0.791 (SE 0.078), H0: θ=1 → z = -2.70, p = 0.0070
θ를 자유롭게 둔 모형의 핵심 계수:


,변화율 차이(%),p
walk_20,-27.1503,0.1177
walk_20_60,0.4395,0.9794
ic_access,-9.3887,0.0098


In [3]:
# 대안 추정법 비교 (같은 X, 노선 FE, 셀 군집)
X = sm.add_constant(design(morning, X_FULL))
groups = pd.factorize(morning["셀 ID"])[0]
rate = morning.y2025 / morning.y2024
alts = {
    "M0 Poisson offset": m0,
    "OLS: 변화율(y25/y24)": sm.OLS(rate, X).fit(cov_type="cluster", cov_kwds={"groups": groups}),
    "OLS: log((y25+1)/(y24+1))": sm.OLS(np.log((morning.y2025 + 1) / (morning.y2024 + 1)), X).fit(cov_type="cluster", cov_kwds={"groups": groups}),
    "WLS: 변화율, 가중치 y24": sm.WLS(rate, X, weights=morning.y2024).fit(cov_type="cluster", cov_kwds={"groups": groups}),
    "음이항 offset (α 추정)": sm.NegativeBinomial(morning.y2025, X, offset=np.log(morning.y2024)).fit(disp=0, maxiter=1000, cov_type="cluster", cov_kwds={"groups": groups}),
}
rows = {}
for name, r in alts.items():
    b, p = r.params["walk_20"], r.pvalues["walk_20"]
    scale = "비율 단위(%p)" if name.startswith("OLS: 변화율") or name.startswith("WLS") else "로그 → %"
    value = b * 100 if scale.startswith("비율") else (np.exp(b) - 1) * 100
    rows[name] = {"보행20분이내 효과": value, "단위": scale, "p": p, "IC p": r.pvalues["ic_access"]}
print("변화율 분포: 중앙값 %.2f, 최대 %.1f (y24 작은 쌍에서 폭주)" % (rate.median(), rate.max()))
pd.DataFrame(rows).T

변화율 분포: 중앙값 0.70, 최대 14.5 (y24 작은 쌍에서 폭주)


,보행20분이내 효과,단위,p,IC p
M0 Poisson offset,-43.756619,로그 → %,0.002769,0.007814
OLS: 변화율(y25/y24),29.583625,비율 단위(%p),0.349922,0.171973
OLS: log((y25+1)/(y24+1)),17.475885,로그 → %,0.573847,0.028085
"WLS: 변화율, 가중치 y24",-41.478722,비율 단위(%p),0.004169,0.005291
음이항 offset (α 추정),11.243791,로그 → %,0.723746,0.009291


### 1-1. θ < 1의 원인 진단: 평균회귀(측정 잡음)인가, 실제 규모 효과인가
하루치 오전 승차는 잡음이 커서, 그 값으로 규모를 재면 θ가 1보다 작게 추정된다(측정오차 감쇠 + 평균회귀).
**같은 셀×노선의 2024년 '오전 외 시간대' 승차**는 오전 잡음과 독립적인 규모 측정치다. 이걸 통제했을 때 효과가 남는지 본다.

In [4]:
other = (trips[~trips["시"].between(*MORNING_HOURS)]
         .pivot_table(index=["셀 ID", "노선"], columns="연도", values="승차인원", aggfunc="sum", fill_value=0)
         .reset_index().rename(columns={2024: "other2024", 2025: "other2025"}))
other["셀 ID"] = other["셀 ID"].astype(int)
mo = morning.merge(other, on=["셀 ID", "노선"], how="left").fillna({"other2024": 0})
mo["log_other2024"] = np.log1p(mo.other2024)
mo["log_walk"] = np.log(mo.walk_min)


def run(frame, x_cols, offset=True, extra=None):
    X = design(frame, x_cols)
    if extra is not None:
        X = pd.concat([X, frame[[extra]]], axis=1)
    model = sm.GLM(frame.y2025, sm.add_constant(X), family=sm.families.Poisson(), offset=np.log(frame.y2024) if offset else None)
    return model.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(frame["셀 ID"])[0]})


mo["log_y2024"] = np.log(mo.y2024)
ad = allday.assign(log_y2024=np.log(allday.y2024))
cases = [
    ("오전 M0 (θ=1)", mo, X_FULL, True, None, "walk_20"),
    ("오전 θ 자유", mo, X_FULL, False, "log_y2024", "walk_20"),
    ("오전 θ=1 + 독립 규모 통제 log(1+2024 오전외)", mo, X_FULL, True, "log_other2024", "walk_20"),
    ("오전 연속 log보행 (θ=1)", mo, ["log_walk"] + X_FULL[2:], True, None, "log_walk"),
    ("오전 연속 log보행 (θ 자유)", mo, ["log_walk"] + X_FULL[2:], False, "log_y2024", "log_walk"),
    ("종일 M0 (θ=1)", ad, X_FULL, True, None, "walk_20"),
    ("종일 θ 자유", ad, X_FULL, False, "log_y2024", "walk_20"),
]
rows = []
for name, frame, cols, offset, extra, key in cases:
    r = run(frame, cols, offset, extra)
    b = r.params[key]
    rows.append({"모형": name, "핵심변수": "보행 20분 이내" if key == "walk_20" else "log 보행시간 (2배당)",
                 "효과(%)": (np.exp(b) - 1) * 100 if key == "walk_20" else (2 ** b - 1) * 100, "p": r.pvalues[key],
                 "IC p": r.pvalues["ic_access"], "추가항 계수": r.params[extra] if extra else np.nan})
pd.DataFrame(rows).round(4)

,모형,핵심변수,효과(%),p,IC p,추가항 계수
0,오전 M0 (θ=1),보행 20분 이내,-43.7566,0.0028,0.0078,NaN
1,오전 θ 자유,보행 20분 이내,-27.1503,0.1177,0.0098,0.7910
2,오전 θ=1 + 독립 규모 통제 log(1+2024 오전외),보행 20분 이내,-39.3537,0.0129,0.0056,-0.0546
3,오전 연속 log보행 (θ=1),log 보행시간 (2배당),27.9363,0.0008,0.0018,NaN
4,오전 연속 log보행 (θ 자유),log 보행시간 (2배당),18.2189,0.0260,0.0032,0.8524
5,종일 M0 (θ=1),보행 20분 이내,-60.3744,0.0000,0.0168,NaN
6,종일 θ 자유,보행 20분 이내,-48.0621,0.0010,0.0071,0.8294


In [5]:
# 승차 규모별 이질성: Poisson은 승차가 많은 쌍에 가중을 크게 준다
rows = []
for label, sub in [("2024 ≥ 20명 쌍", morning[morning.y2024 >= 20]), ("2024 < 20명 쌍", morning[morning.y2024 < 20])]:
    sub = sub.reset_index(drop=True)
    r = fit(sub, X_FULL)
    rows.append({"부분표본": label, "쌍": len(sub), "20분 이내 쌍": int(sub.walk_20.sum()), "2024 승차 비중(%)": sub.y2024.sum() / morning.y2024.sum() * 100, **effect(r, "walk_20")})
pd.DataFrame(rows).round(3)

,부분표본,쌍,20분 이내 쌍,2024 승차 비중(%),변화율 차이(%),p
0,2024 ≥ 20명 쌍,87,16,85.705,-53.028,0.000
1,2024 < 20명 쌍,99,7,14.295,588.051,0.001


## 2. 보행시간 경계(20분/60분) 민감도

- 단일 경계: 기준 이내 = 1 (나머지 전체가 기준)
- 3구간: 여러 경계 조합
- 연속형: 선형, log

In [6]:
x_controls = ["ic_access", "competing_share", "pop_20_50", "buildings"]
rows = []
for frame, period in [(morning, "오전"), (allday, "종일")]:
    for cutoff in (10, 15, 20, 25, 30, 40, 60):
        f = frame.assign(near=(frame.walk_min <= cutoff).astype(float))
        r = fit(f, ["near"] + x_controls)
        rows.append({"시간대": period, "기준(분)": cutoff, "기준 이내 셀": f.loc[f.near == 1, "셀 ID"].nunique(), **effect(r, "near")})
single = pd.DataFrame(rows)
single.pivot(index=["기준(분)"], columns="시간대", values=["기준 이내 셀", "변화율 차이(%)", "p"]).round(4)

기준 이내 셀       변화율 차이(%)                p        
시간대        오전    종일        오전       종일      오전      종일
기준(분)                                                 
10        3.0   3.0  -43.0110 -59.5331  0.0167  0.0000
15        3.0   3.0  -43.0110 -59.5331  0.0167  0.0000
20       11.0  11.0  -32.0404 -47.5666  0.0075  0.0000
25       16.0  16.0  -39.0600 -29.3227  0.0002  0.0008
30       26.0  26.0  -20.9450 -20.6895  0.1491  0.0349
40       53.0  54.0  -27.0749 -25.0778  0.0855  0.0014
60       81.0  85.0  -16.8763 -26.9439  0.2019  0.0099

In [7]:
rows = []
for lo, hi in [(10, 30), (15, 45), (15, 60), (20, 40), (20, 60), (30, 60), (30, 90)]:
    for frame, period in [(morning, "오전"), (allday, "종일")]:
        f = frame.assign(a=(frame.walk_min <= lo).astype(float), b=((frame.walk_min > lo) & (frame.walk_min <= hi)).astype(float))
        r = fit(f, ["a", "b"] + x_controls)
        rows.append({"구간": f"≤{lo} / {lo}~{hi} / >{hi}", "시간대": period, "n(≤lo 셀)": f.loc[f.a == 1, "셀 ID"].nunique(),
                     "≤lo 효과(%)": effect(r, "a")["변화율 차이(%)"], "≤lo p": r.pvalues["a"],
                     "중간 효과(%)": effect(r, "b")["변화율 차이(%)"], "중간 p": r.pvalues["b"]})
pd.DataFrame(rows).round(3)

,구간,시간대,n(≤lo 셀),≤lo 효과(%),≤lo p,중간 효과(%),중간 p
0,≤10 / 10~30 / >30,오전,3,-48.169,0.019,-15.338,0.303
1,≤10 / 10~30 / >30,종일,3,-61.913,0.000,-8.783,0.411
2,≤15 / 15~45 / >45,오전,3,-37.240,0.050,11.989,0.500
3,≤15 / 15~45 / >45,종일,3,-61.587,0.000,-6.097,0.585
4,≤15 / 15~60 / >60,오전,3,-53.259,0.003,-17.800,0.138
5,≤15 / 15~60 / >60,종일,3,-68.929,0.000,-24.181,0.025
6,≤20 / 20~40 / >40,오전,11,-44.638,0.006,-22.211,0.156
7,≤20 / 20~40 / >40,종일,11,-54.890,0.000,-16.361,0.041
8,≤20 / 20~60 / >60,오전,11,-43.757,0.003,-17.195,0.163
9,≤20 / 20~60 / >60,종일,11,-60.374,0.000,-24.946,0.014


In [8]:
rows = []
for frame, period in [(morning, "오전"), (allday, "종일")]:
    f = frame.assign(log_walk=np.log(frame.walk_min), walk10=frame.walk_min / 10)
    for name, col, label in [("선형 (10분당)", "walk10", "10분 늘 때 변화율"), ("log (2배당)", "log_walk", "보행시간 2배 → 변화율")]:
        r = fit(f, [col] + x_controls)
        b = r.params[col]
        rows.append({"시간대": period, "형태": name, "효과(%)": ((2 ** b) - 1) * 100 if col == "log_walk" else (np.exp(b) - 1) * 100, "p": r.pvalues[col]})
pd.DataFrame(rows).round(4)

,시간대,형태,효과(%),p
0,오전,선형 (10분당),5.4531,0.0361
1,오전,log (2배당),27.9363,0.0008
2,종일,선형 (10분당),6.2907,0.0017
3,종일,log (2배당),32.2438,0.0001


## 3. 2024년 승차 0인 셀·쌍 제외의 선택편향

In [9]:
cell_morning = data.assign(포함=data.board_2024 > 0)
compare = cell_morning.groupby("포함")[["walk_min", "ic_access", "pop_20_50", "buildings", "board_2025", "board_2024_all", "board_2025_all"]].agg(["mean", "median"]).T
compare.columns = ["제외(2024 오전 0)", "포함"]
display(compare.round(1))
print("제외 셀의 보행시간 구간:", cell_morning.loc[~cell_morning.포함, "walk_group"].value_counts().to_dict())
print("제외 셀의 2025 오전 승차 합계:", cell_morning.loc[~cell_morning.포함, "board_2025"].sum())

zero_pairs = build_pairs(MORNING_HOURS, keep_zero=True)
print(f"셀×노선 전체 {len(zero_pairs)}쌍 중 2024=0: {(zero_pairs.y2024 == 0).sum()}쌍 (그중 2025>0: {((zero_pairs.y2024 == 0) & (zero_pairs.y2025 > 0)).sum()}쌍, 2025 승차 합 {zero_pairs.loc[zero_pairs.y2024 == 0, 'y2025'].sum():.0f}명)")

제외(2024 오전 0)     포함
walk_min       mean            100.5   63.1
               median           87.0   44.2
ic_access      mean              4.1    5.1
               median            2.8    4.8
pop_20_50      mean            319.8  687.2
               median           37.0  607.0
buildings      mean             33.6   32.7
               median           19.5   23.0
board_2025     mean              1.4   40.1
               median            0.0   20.0
board_2024_all mean              5.9  167.8
               median            4.0   77.5
board_2025_all mean             11.6  136.8
               median            7.0   65.0

제외 셀의 보행시간 구간: {'60분 초과': 10, '20~60분': 4, '20분 이내': 0}
제외 셀의 2025 오전 승차 합계: 20.0
셀×노선 전체 201쌍 중 2024=0: 15쌍 (그중 2025>0: 15쌍, 2025 승차 합 75명)


### 0을 포함하는 대안: 쌍(셀×노선) 고정효과 Poisson 패널
연도별 2행(2024, 2025)으로 펼쳐 `log E[y_irt] = α_ir + post_t·(Xβ + γ_r)`를 추정한다. 쌍 고정효과 α_ir가 수준을 흡수하므로 offset 모형과 같은 변화율 효과를 추정한다. 이때 **2024=0, 2025>0인 쌍도 정보에 들어간다**(두 해 모두 0인 쌍만 빠짐).

In [10]:
long = zero_pairs[(zero_pairs.y2024 + zero_pairs.y2025) > 0].reset_index(drop=True)
long["pair"] = np.arange(len(long))
panel = pd.concat([long.assign(y=long.y2024, post=0.0), long.assign(y=long.y2025, post=1.0)], ignore_index=True)
Xp = design(panel, X_FULL).mul(panel.post, axis=0)
Xp = pd.concat([Xp, pd.get_dummies(panel.pair, prefix="pair", dtype=float)], axis=1)
Xp["post"] = panel.post
panel_fit = sm.GLM(panel.y, Xp, family=sm.families.Poisson()).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(panel["셀 ID"])[0]})
print(f"패널: 쌍 {len(long)}개 (2024=0 포함 {(long.y2024 == 0).sum()}쌍)")
pd.DataFrame({"M0 (2024>0만)": [effect(m0, n)["변화율 차이(%)"] for n in ["walk_20", "walk_20_60", "ic_access"]],
              "M0 p": [m0.pvalues[n] for n in ["walk_20", "walk_20_60", "ic_access"]],
              "쌍FE 패널 (0 포함)": [effect(panel_fit, n)["변화율 차이(%)"] for n in ["walk_20", "walk_20_60", "ic_access"]],
              "패널 p": [panel_fit.pvalues[n] for n in ["walk_20", "walk_20_60", "ic_access"]]},
             index=["보행 20분 이내", "보행 20~60분", "IC 도로거리"]).round(4)

패널: 쌍 201개 (2024=0 포함 15쌍)


,M0 (2024>0만),M0 p,쌍FE 패널 (0 포함),패널 p
보행 20분 이내,-43.7566,0.0028,-45.5934,0.0287
보행 20~60분,-17.1945,0.1632,-22.5071,0.1984
IC 도로거리,-10.7675,0.0078,-11.5937,0.0166


## 4. 고정효과 구조 비교

In [11]:
rows = []
for frame, period in [(morning, "오전"), (allday, "종일")]:
    for label, rfe, sfe in [("FE 없음", False, False), ("노선 FE", True, False), ("역 FE", False, True), ("노선 FE + 역 FE", True, True)]:
        r = fit(frame, X_FULL, route_fe=rfe, station_fe=sfe)
        rows.append({"시간대": period, "고정효과": label,
                     **{f"{VARIABLE_LABELS[n]} (%)": effect(r, n)["변화율 차이(%)"] for n in ["walk_20", "walk_20_60", "ic_access"]},
                     **{f"{VARIABLE_LABELS[n]} p": r.pvalues[n] for n in ["walk_20", "walk_20_60", "ic_access"]}})
pd.DataFrame(rows).round(4)

,시간대,고정효과,보행 20분 이내 (%),보행 20~60분 (%),IC까지 도로거리(km) (%),보행 20분 이내 p,보행 20~60분 p,IC까지 도로거리(km) p
0,오전,FE 없음,-41.0341,-21.1750,-9.0418,0.0055,0.1861,0.0037
1,오전,노선 FE,-43.7566,-17.1945,-10.7675,0.0028,0.1632,0.0078
2,오전,역 FE,-30.6520,-19.8302,-5.9328,0.0429,0.1910,0.0357
3,오전,노선 FE + 역 FE,-33.3274,-9.2383,-7.7188,0.0585,0.5856,0.0302
4,종일,FE 없음,-50.4714,-28.8465,-5.1690,0.0001,0.0151,0.0275
5,종일,노선 FE,-60.3744,-24.9457,-5.6819,0.0000,0.0142,0.0168
6,종일,역 FE,-52.2872,-31.8485,-3.7909,0.0001,0.0079,0.0872
7,종일,노선 FE + 역 FE,-60.6108,-28.3761,-3.4578,0.0000,0.0061,0.1105


In [12]:
# 노선 FE가 흡수하는 '노선 간' GTX 노출: 노선별 20분 이내 쌍 비중과 노선 FE 크기
route_fx = m0.params.filter(like="노선_")
exposure = morning.groupby("노선").apply(lambda g: np.average(g.walk_20, weights=g.y2024)).rename("20분 이내 승차 비중")
base_route = sorted(morning.노선.unique())[0]
fx = pd.concat([exposure, route_fx.rename(lambda s: s.replace("노선_", "")).rename("노선 FE")], axis=1).fillna({"노선 FE": 0.0})
rho, p = spearmanr(fx["20분 이내 승차 비중"], fx["노선 FE"])
print(f"기준 노선 {base_route}. 노선별 GTX 근접 노출과 노선 FE의 Spearman = {rho:.2f} (p={p:.3f})")
fx.sort_values("노선 FE").round(3)

기준 노선 1000. 노선별 GTX 근접 노출과 노선 FE의 Spearman = 0.06 (p=0.772)


,20분 이내 승차 비중,노선 FE
1100,0.000,-22.221
M7111,0.000,-0.690
2200,0.000,-0.687
M7154,0.434,-0.580
1500,0.109,-0.568
9700,0.000,-0.476
1900,0.000,-0.462
G7625,0.413,-0.366
9600,0.034,-0.275
3400,0.505,-0.174


## 5. 공간자기상관 (M0 잔차)

In [13]:
centers = stable_centers().set_index("셀 ID")
cells = morning.assign(mu=m0.mu).groupby("셀 ID")[["y2025", "mu"]].sum()
cells["resid"] = (cells.y2025 - cells.mu) / np.sqrt(cells.mu)  # 셀 단위 Pearson 잔차
xy = np.c_[centers.loc[cells.index].geometry.x, centers.loc[cells.index].geometry.y]
D = np.sqrt(((xy[:, None] - xy[None]) ** 2).sum(-1))


def morans_i(values, W):
    z = values - values.mean()
    return len(z) / W.sum() * (z @ W @ z) / (z @ z)


rng = np.random.default_rng(0)
rows = []
for label, W in [("1km 이내", ((D > 0) & (D <= 1000)).astype(float)), ("2km 이내", ((D > 0) & (D <= 2000)).astype(float)),
                 ("최근접 5개", (D <= np.sort(D, axis=1)[:, [5]]).astype(float) - np.eye(len(D)))]:
    W = W / np.where(W.sum(1, keepdims=True) > 0, W.sum(1, keepdims=True), 1)
    obs = morans_i(cells.resid.values, W)
    sims = np.array([morans_i(rng.permutation(cells.resid.values), W) for _ in range(999)])
    rows.append({"가중치": label, "Moran's I": obs, "순열 p": (np.sum(sims >= obs) + 1) / 1000})
pd.DataFrame(rows).round(3)

,가중치,Moran's I,순열 p
0,1km 이내,0.075,0.180
1,2km 이내,0.019,0.290
2,최근접 5개,0.015,0.307


In [14]:
# Conley 공간 HAC (셀×노선 행, 셀 간 거리, Bartlett) + 읍면동 군집
def conley_p(frame, result, cutoff_m):
    Xv = sm.add_constant(design(frame, X_FULL)).values
    mu = np.asarray(result.mu)
    score = Xv * (frame.y2025.values - mu)[:, None]
    bread = np.linalg.pinv(Xv.T @ (Xv * mu[:, None]))
    cxy = np.c_[centers.loc[frame["셀 ID"]].geometry.x, centers.loc[frame["셀 ID"]].geometry.y]
    dist = np.sqrt(((cxy[:, None] - cxy[None]) ** 2).sum(-1))
    se = np.sqrt(np.diag(bread @ (score.T @ np.clip(1 - dist / cutoff_m, 0, None) @ score) @ bread))
    names = ["const"] + list(design(frame, X_FULL).columns)
    return pd.Series(2 * norm.sf(np.abs(result.params.values / se)), index=names)


def fix_korean(text):
    try:
        return str(text).encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return str(text)


emd = read_csv(IC_PATH)[["셀 ID", "emd_nm_k"]].assign(읍면동=lambda d: d.emd_nm_k.map(fix_korean))[["셀 ID", "읍면동"]]
m_emd = morning.merge(emd, on="셀 ID")
r_emd = fit(m_emd, X_FULL, cluster="읍면동")
table = pd.DataFrame({"셀 군집 (M0)": m0.pvalues, "읍면동 군집": r_emd.pvalues,
                      "Conley 1km": conley_p(morning, m0, 1000), "Conley 2km": conley_p(morning, m0, 2000), "Conley 5km": conley_p(morning, m0, 5000)})
table.loc[["walk_20", "walk_20_60", "ic_access"]].rename(index=VARIABLE_LABELS).round(4)

,셀 군집 (M0),읍면동 군집,Conley 1km,Conley 2km,Conley 5km
보행 20분 이내,0.0028,0.0023,0.0014,0.0011,0.0003
보행 20~60분,0.1632,0.2052,0.1343,0.1491,0.1304
IC까지 도로거리(km),0.0078,0.0409,0.0065,0.0093,0.0121


## 6. 독립 정보량과 영향 관측치

In [15]:
near = morning[morning.walk_20 == 1]
print(f"보행 20분 이내: 셀 {near['셀 ID'].nunique()}개, 쌍 {len(near)}개, 노선 {near.노선.nunique()}개, 2024 승차 {near.y2024.sum():.0f}명 (전체의 {near.y2024.sum()/morning.y2024.sum()*100:.0f}%)")
print("역별 20분 이내 셀:", near.groupby("nearest_station")["셀 ID"].nunique().to_dict())
print("20분 이내 셀별 2024 승차 비중(%):", (near.groupby("셀 ID").y2024.sum() / near.y2024.sum() * 100).round(1).sort_values(ascending=False).to_dict())

# 노선 FE 하에서 walk_20 식별에 기여하는 노선: 같은 노선 안에 20분 이내 쌍과 아닌 쌍이 모두 있는 노선
contrib = morning.groupby("노선").agg(near=("walk_20", "sum"), n=("walk_20", "size"))
print("노선 내 대비가 있는 노선(20분 이내와 그 외 둘 다 있음):", contrib[(contrib.near > 0) & (contrib.near < contrib.n)].index.tolist())

보행 20분 이내: 셀 11개, 쌍 23개, 노선 11개, 2024 승차 1158명 (전체의 19%)
역별 20분 이내 셀: {'대곡': 1, '운정중앙': 6, '킨텍스': 4}
20분 이내 셀별 2024 승차 비중(%): {5217: 22.3, 12765: 20.3, 11239: 18.7, 13084: 13.0, 14103: 6.8, 7697: 5.7, 13602: 5.0, 8868: 4.4, 3765: 2.3, 14783: 1.5, 8546: 0.1}
노선 내 대비가 있는 노선(20분 이내와 그 외 둘 다 있음): ['1000', '1500', '3100', '3400', '7101', '9030', '9600', 'G7625', 'M7154', 'M7731']


In [16]:
rows = []
for station in morning.nearest_station.unique():
    f = morning[morning.nearest_station != station]
    r = fit(f, X_FULL)
    rows.append({"뺀 것": f"역 권역: {station}", "남은 20분 이내 셀": f.loc[f.walk_20 == 1, "셀 ID"].nunique(), **effect(r, "walk_20")})
for cell in near["셀 ID"].unique():
    f = morning[morning["셀 ID"] != cell]
    rows.append({"뺀 것": f"20분 이내 셀 {cell}", "남은 20분 이내 셀": 10, **effect(fit(f, X_FULL), "walk_20")})
for route in contrib[(contrib.near > 0)].index:
    f = morning[morning.노선 != route]
    rows.append({"뺀 것": f"노선 {route}", "남은 20분 이내 셀": f.loc[f.walk_20 == 1, "셀 ID"].nunique(), **effect(fit(f, X_FULL), "walk_20")})
influence = pd.DataFrame(rows)
influence.round(4)

,뺀 것,남은 20분 이내 셀,변화율 차이(%),p
0,역 권역: 대곡,10,-47.0079,0.0002
1,역 권역: 운정중앙,5,-50.2798,0.1011
2,역 권역: 킨텍스,7,-18.3117,0.4391
3,20분 이내 셀 3765,10,-44.4166,0.0025
4,20분 이내 셀 5217,10,-45.0789,0.0016
5,20분 이내 셀 7697,10,-43.2815,0.0037
6,20분 이내 셀 8546,10,-44.3196,0.0023
7,20분 이내 셀 8868,10,-39.8553,0.0166
8,20분 이내 셀 11239,10,-42.7431,0.0070
9,20분 이내 셀 12765,10,-39.1602,0.0132


In [17]:
# 전체 118개 셀 leave-one-cell-out: walk_20 계수 분포
loo = []
for cell in morning["셀 ID"].unique():
    r = fit(morning[morning["셀 ID"] != cell], X_FULL)
    loo.append({"셀 ID": cell, "효과(%)": effect(r, "walk_20")["변화율 차이(%)"], "p": r.pvalues["walk_20"]})
loo = pd.DataFrame(loo)
print(f"leave-one-cell-out (118회): 효과 {loo['효과(%)'].min():.1f} ~ {loo['효과(%)'].max():.1f}%, p 최대 {loo.p.max():.4f}, p>0.05 횟수 {(loo.p > 0.05).sum()}")
loo.sort_values("p", ascending=False).head(5).round(4)

leave-one-cell-out (118회): 효과 -48.2 ~ -39.2%, p 최대 0.0211, p>0.05 횟수 0


,셀 ID,효과(%),p
102,13315,-41.3709,0.0211
56,8868,-39.8553,0.0166
94,12765,-39.1602,0.0132
117,15474,-41.7297,0.0080
55,8846,-42.0625,0.0076


## 7. 시간대 선택 (오전 vs 종일 vs 다른 시간대)

In [18]:
rows = []
for label, hours in [("오전 07~09 (본)", (7, 9)), ("출근 06~09", (6, 9)), ("오전 07~08", (7, 8)), ("낮 10~15", (10, 15)), ("저녁 16~23", (16, 23)), ("종일", (0, 23))]:
    f = build_pairs(hours)
    r = fit(f, X_FULL)
    rows.append({"시간대": label, "N(쌍)": len(f), "셀": f["셀 ID"].nunique(), **effect(r, "walk_20"),
                 "IC 효과(%)": effect(r, "ic_access")["변화율 차이(%)"], "IC p": r.pvalues["ic_access"]})
pd.DataFrame(rows).round(4)

,시간대,N(쌍),셀,변화율 차이(%),p,IC 효과(%),IC p
0,오전 07~09 (본),186,118,-43.7566,0.0028,-10.7675,0.0078
1,출근 06~09,196,121,-52.2205,0.0001,-8.2355,0.0165
2,오전 07~08,181,115,-40.1606,0.0129,-8.3593,0.0517
3,낮 10~15,158,104,-71.7734,0.0001,-10.4105,0.0046
4,저녁 16~23,161,100,-41.5431,0.0533,-0.6435,0.8767
5,종일,220,132,-60.3744,0.0000,-5.6819,0.0168


## 8. pseudo R²

`10_최종모형`의 0.42 = **deviance 기반 pseudo R²** = 1 − D(모형) / D(절편+offset만).
노선 FE의 설명분이 섞여 있으므로, 노선 FE만 넣은 모형과 비교해 **X의 추가 설명분**을 분리한다.

In [19]:
def deviance(frame, X):
    return sm.GLM(frame.y2025, sm.add_constant(X) if X is not None else np.ones((len(frame), 1)),
                  family=sm.families.Poisson(), offset=np.log(frame.y2024)).fit().deviance


rows = []
for frame, period in [(morning, "오전"), (allday, "종일")]:
    d_null = deviance(frame, None)
    d_route = deviance(frame, design(frame, [], route_fe=True))
    d_x = deviance(frame, frame[X_FULL])
    d_full = deviance(frame, design(frame, X_FULL))
    rows.append({"시간대": period,
                 "R²_D 전체 (=10번 보고값)": 1 - d_full / d_null,
                 "R²_D 노선 FE만": 1 - d_route / d_null,
                 "R²_D X만 (FE 없음)": 1 - d_x / d_null,
                 "노선 FE 위에 X가 추가 설명 (부분 R²_D)": 1 - d_full / d_route})
pd.DataFrame(rows).round(3)

,시간대,R²_D 전체 (=10번 보고값),R²_D 노선 FE만,R²_D X만 (FE 없음),노선 FE 위에 X가 추가 설명 (부분 R²_D)
0,오전,0.424,0.346,0.090,0.119
1,종일,0.478,0.296,0.132,0.258


## 9. 변수 구성 대안

In [20]:
specs = {
    "전체 (M0)": X_FULL,
    "핵심: 보행 + IC": ["walk_20", "walk_20_60", "ic_access"],
    "핵심 + 경쟁목적지": ["walk_20", "walk_20_60", "ic_access", "competing_share"],
    "핵심 + 인구": ["walk_20", "walk_20_60", "ic_access", "pop_20_50"],
    "보행만": ["walk_20", "walk_20_60"],
}
rows = []
for frame, period in [(morning, "오전"), (allday, "종일")]:
    for name, cols in specs.items():
        r = fit(frame, cols)
        rows.append({"시간대": period, "변수 구성": name, **{f"20분 이내 {k}": v for k, v in effect(r, "walk_20").items()},
                     "IC (%)": effect(r, "ic_access")["변화율 차이(%)"] if "ic_access" in cols else np.nan,
                     "IC p": r.pvalues.get("ic_access", np.nan)})
pd.DataFrame(rows).round(4)

,시간대,변수 구성,20분 이내 변화율 차이(%),20분 이내 p,IC (%),IC p
0,오전,전체 (M0),-43.7566,0.0028,-10.7675,0.0078
1,오전,핵심: 보행 + IC,-44.4394,0.0008,-10.3297,0.0046
2,오전,핵심 + 경쟁목적지,-45.0703,0.0007,-10.4113,0.0047
3,오전,핵심 + 인구,-43.0799,0.0034,-10.2782,0.0049
4,오전,보행만,-25.5362,0.0865,NaN,NaN
5,종일,전체 (M0),-60.3744,0.0000,-5.6819,0.0168
6,종일,핵심: 보행 + IC,-59.2328,0.0000,-4.7992,0.0272
7,종일,핵심 + 경쟁목적지,-59.7118,0.0000,-4.9118,0.0266
8,종일,핵심 + 인구,-59.8654,0.0000,-5.3543,0.0134
9,종일,보행만,-54.9333,0.0003,NaN,NaN


In [21]:
print("변수 간 상관 (셀 132개):")
data[["walk_min", "ic_access", "competing_share", "pop_20_50", "buildings", "board_2024"]].corr().round(2)

변수 간 상관 (셀 132개):


,walk_min,ic_access,competing_share,pop_20_50,buildings,board_2024
walk_min,1.00,0.42,0.38,-0.41,0.20,-0.34
ic_access,0.42,1.00,0.33,-0.06,-0.12,-0.10
competing_share,0.38,0.33,1.00,-0.05,0.15,-0.01
pop_20_50,-0.41,-0.06,-0.05,1.00,-0.08,0.19
buildings,0.20,-0.12,0.15,-0.08,1.00,-0.15
board_2024,-0.34,-0.10,-0.01,0.19,-0.15,1.00


---
**판정과 수정 사항은 `../방법론_감사.md`에 정리했다.**